In [1]:
import os

BASE_PATH = "/kaggle/input/datasets/idkabtit/violencepredictiondataset/kaggleDataset/"

print(os.listdir(BASE_PATH))

['violenceFrame', 'nonViolenceFrame']


In [2]:
print(os.listdir("/kaggle/input/datasets/idkabtit/violencepredictiondataset/kaggleDataset"))

['violenceFrame', 'nonViolenceFrame']


In [3]:
print(os.listdir("/kaggle/input/datasets/idkabtit/violencepredictiondataset/kaggleDataset/violenceFrame"))

['video18', 'video10', 'video22', 'video21', 'video13', 'video9', 'video6', 'video47', 'video16', 'dataset_kaggle.csv', 'video11', 'video8', 'video3', 'video30', 'video1', 'video4', 'video7', 'video14', 'video34', 'video5', 'video31', 'video12', 'video2', 'video15']


In [4]:
import os

BASE_PATH = "/kaggle/input/datasets/idkabtit/violencepredictiondataset/kaggleDataset/"

def count_frames(folder_path, name):
    print(f"\n===== {name} =====")

    for video in sorted(os.listdir(folder_path)):
        video_path = os.path.join(folder_path, video)

        if not os.path.isdir(video_path):
            continue

        frames = [
            f for f in os.listdir(video_path)
            if f.endswith(".jpg") or f.endswith(".png")
        ]

        print(f"{video}: {len(frames)} frames")


# 🔥 Violence videos
count_frames(BASE_PATH + "violenceFrame", "VIOLENCE")

# 🔥 Non-violence videos
count_frames(BASE_PATH + "nonViolenceFrame", "NON-VIOLENCE")


===== VIOLENCE =====
video1: 288 frames
video10: 865 frames
video11: 492 frames
video12: 1512 frames
video13: 216 frames
video14: 1320 frames
video15: 192 frames
video16: 8406 frames
video18: 192 frames
video2: 417 frames
video21: 876 frames
video22: 2590 frames
video3: 300 frames
video30: 312 frames
video31: 456 frames
video34: 313 frames
video4: 348 frames
video47: 380 frames
video5: 168 frames
video6: 84 frames
video7: 228 frames
video8: 312 frames
video9: 420 frames

===== NON-VIOLENCE =====
video1: 80 frames
video10: 80 frames
video11: 80 frames
video12: 80 frames
video13: 80 frames
video14: 80 frames
video15: 80 frames
video16: 80 frames
video17: 80 frames
video18: 80 frames
video19: 80 frames
video2: 80 frames
video20: 80 frames
video21: 80 frames
video22: 80 frames
video23: 80 frames
video24: 80 frames
video25: 80 frames
video26: 80 frames
video27: 80 frames
video28: 80 frames
video29: 80 frames
video3: 80 frames
video30: 80 frames
video31: 80 frames
video32: 80 frames
video33

In [5]:
import os
import re

def check_missing_frames(folder_path):
    print("\n===== CHECK MISSING FRAMES =====")

    total_missing = 0
    videos_checked = 0

    for video in sorted(os.listdir(folder_path)):
        video_path = os.path.join(folder_path, video)

        if not os.path.isdir(video_path):
            continue

        videos_checked += 1

        frames = sorted([
            f for f in os.listdir(video_path)
            if f.endswith(".jpg") or f.endswith(".png")
        ])

        if not frames:
            print(f"⚠️ {video}: No frames found")
            continue

        numbers = []
        for f in frames:
            match = re.search(r'(\d+)', f)
            if match:
                numbers.append(int(match.group()))

        if not numbers:
            print(f"⚠️ {video}: No valid frame numbers")
            continue

        missing = []
        for i in range(min(numbers), max(numbers)):
            if i not in numbers:
                missing.append(i)

        if missing:
            total_missing += len(missing)
            print(f"❌ {video}: Missing {len(missing)} frames (e.g., {missing[:5]})")

    # ✅ Final summary
    print("\n===== SUMMARY =====")
    print(f"Videos checked: {videos_checked}")
    print(f"Total missing frames: {total_missing}")

    if total_missing == 0:
        print("✅ No missing frames found. Dataset is clean!")
    else:
        print("⚠️ Missing frames detected. Dataset needs fixing.")


# 🔥 RUN (change path accordingly)
check_missing_frames("/kaggle/input/datasets/idkabtit/violencepredictiondataset/kaggleDataset/violenceFrame")


===== CHECK MISSING FRAMES =====

===== SUMMARY =====
Videos checked: 23
Total missing frames: 0
✅ No missing frames found. Dataset is clean!


In [6]:
import pandas as pd
from sklearn.utils import shuffle

BASE_PATH = "/kaggle/input/datasets/idkabtit/violencepredictiondataset/kaggleDataset/"

df_v = pd.read_csv(BASE_PATH + "violenceFrame/dataset_kaggle.csv")
df_nv = pd.read_csv(BASE_PATH + "nonViolenceFrame/dataset_kaggle.csv")

df = pd.concat([df_v, df_nv], ignore_index=True)
df = shuffle(df, random_state=42).reset_index(drop=True)

print("Total samples:", len(df))
print(df["label"].value_counts())

Total samples: 5985
label
0    3027
1    2958
Name: count, dtype: int64


In [7]:
import os
import ast
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# ================= CONFIG =================
ROOT = "/kaggle/input/datasets/idkabtit/violencepredictiondataset/kaggleDataset"
VIOLENCE_DIR    = os.path.join(ROOT, "violenceFrame")
NONVIOLENCE_DIR = os.path.join(ROOT, "nonViolenceFrame")
CSV_V  = os.path.join(VIOLENCE_DIR, "dataset_kaggle.csv")
CSV_NV = os.path.join(NONVIOLENCE_DIR, "dataset_kaggle.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

WINDOW_SIZE = 16
BATCH_SIZE  = 2

# ================= TRANSFORM =================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ================= DATASET =================
class ViolenceDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        source = row["source"]
        label  = float(row["label"])

        base_dir = VIOLENCE_DIR if source == "violence" else NONVIOLENCE_DIR

        raw = str(row["frame_paths"])  # ✅ safe cast
        if "|" in raw:
            paths = raw.split("|")
        else:
            try:
                paths = ast.literal_eval(raw)
            except:
                paths = raw.split(",")

        frames = []
        for p in paths:
            p = p.strip()
            full_path = os.path.join(base_dir, p)
            if not os.path.exists(full_path):
                raise Exception(f"❌ Missing file: {full_path}")
            img = Image.open(full_path).convert("RGB")
            img = transform(img)
            frames.append(img)

        if len(frames) != WINDOW_SIZE:
            raise Exception(f"❌ Window size mismatch at index {idx}: got {len(frames)}")

        return torch.stack(frames), torch.tensor(label, dtype=torch.float32)

# ================= LOAD DATA =================
df_v  = pd.read_csv(CSV_V);  df_v["source"]  = "violence"
df_nv = pd.read_csv(CSV_NV); df_nv["source"] = "nonviolence"

df = pd.concat([df_v, df_nv], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df.head(20)

print("Sanity samples:", len(df))
print(df["label"].value_counts())

dataset = ViolenceDataset(df)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# ================= MODEL =================
class CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.cnn  = nn.Sequential(*list(resnet.children())[:-1])
        self.lstm = nn.LSTM(512, 128, batch_first=True)
        self.fc   = nn.Linear(128, 1)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x    = x.view(B * T, C, H, W)
        feat = self.cnn(x).view(B, T, 512)
        out, _ = self.lstm(feat)
        return self.fc(out[:, -1, :]).squeeze(1)

model     = CNN_LSTM().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ================= SANITY RUN =================
print("\n🚀 Starting SANITY RUN...\n")
model.train()

for i, (frames, labels) in enumerate(loader):
    frames = frames.to(DEVICE)
    labels = labels.to(DEVICE)

    optimizer.zero_grad()
    logits = model(frames)
    loss   = criterion(logits, labels)
    loss.backward()
    optimizer.step()

    print(f"Batch {i} | Loss: {loss.item():.4f}")

    if i == 2:
        break

print("\n✅ SANITY RUN COMPLETE")

Device: cuda
Sanity samples: 20
label
0    10
1    10
Name: count, dtype: int64
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 213MB/s]



🚀 Starting SANITY RUN...

Batch 0 | Loss: 0.6020
Batch 1 | Loss: 0.6532
Batch 2 | Loss: 0.9067

✅ SANITY RUN COMPLETE


In [8]:
# ============================================================
# 🚀 EARLY VIOLENCE PREDICTION (CNN + LSTM)
# ============================================================

import os
import ast
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score

# ------------------ CONFIG ------------------
ROOT = "/kaggle/input/datasets/idkabtit/violencepredictiondataset/kaggleDataset"

VIOLENCE_DIR    = os.path.join(ROOT, "violenceFrame")
NONVIOLENCE_DIR = os.path.join(ROOT, "nonViolenceFrame")

CSV_V  = os.path.join(VIOLENCE_DIR, "dataset_kaggle.csv")
CSV_NV = os.path.join(NONVIOLENCE_DIR, "dataset_kaggle.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WINDOW_SIZE = 16
BATCH_SIZE  = 16
EPOCHS      = 20
LR          = 3e-5    # ✅ lowered from 1e-4

print("Device:", DEVICE)
print("GPUs available:", torch.cuda.device_count())

# ------------------ TRANSFORM ------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ============================================================
# 📦 DATASET
# ============================================================
class ViolenceDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        source = row["source"]
        label  = float(row["label"])

        base_dir = VIOLENCE_DIR if source == "violence" else NONVIOLENCE_DIR

        raw = str(row["frame_paths"])
        if "|" in raw:
            paths = raw.split("|")
        else:
            try:
                paths = ast.literal_eval(raw)
            except:
                paths = raw.split(",")

        frames = []

        for p in paths:
            p = p.strip()
            full_path = os.path.join(base_dir, p)

            if not os.path.exists(full_path):
                raise Exception(f"Missing file: {full_path}")

            img = Image.open(full_path).convert("RGB")
            img = transform(img)
            frames.append(img)

        if len(frames) != WINDOW_SIZE:
            raise Exception(f"Window size mismatch at index {idx}")

        return torch.stack(frames), torch.tensor(label, dtype=torch.float32)

# ============================================================
# 📊 LOAD DATA
# ============================================================
df_v  = pd.read_csv(CSV_V);  df_v["source"]  = "violence"
df_nv = pd.read_csv(CSV_NV); df_nv["source"] = "nonviolence"

# make video_id unique per source (in memory only)
df_v["video_id"]  = "v_"  + df_v["video_id"].astype(str)
df_nv["video_id"] = "nv_" + df_nv["video_id"].astype(str)

df = pd.concat([df_v, df_nv], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nFull dataset distribution:")
print(df["label"].value_counts())

# split by video
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df["video_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

# confirm no leakage
train_videos = set(train_df["video_id"])
val_videos   = set(val_df["video_id"])
overlap      = train_videos & val_videos
print(f"\nTrain videos: {len(train_videos)} | Val videos: {len(val_videos)}")
print(f"Overlap (should be 0): {len(overlap)}")
assert len(overlap) == 0, "❌ Data leakage detected!"
print("✅ No leakage confirmed")

print("\n=== Train label distribution ===")
print(train_df["label"].value_counts())
print("\n=== Val label distribution ===")
print(val_df["label"].value_counts())

train_dataset = ViolenceDataset(train_df)
val_dataset   = ViolenceDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ============================================================
# 🧠 MODEL (with Dropout)
# ============================================================
class CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()

        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.cnn     = nn.Sequential(*list(resnet.children())[:-1])
        self.lstm    = nn.LSTM(512, 128, batch_first=True)
        self.dropout = nn.Dropout(0.5)        # ✅ added
        self.fc      = nn.Linear(128, 1)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x        = x.view(B * T, C, H, W)
        features = self.cnn(x).view(B, T, 512)
        out, _   = self.lstm(features)
        out      = self.dropout(out[:, -1, :])  # ✅ dropout before FC
        return self.fc(out).squeeze(1)

# ============================================================
# ⚙️ SETUP
# ============================================================
model = CNN_LSTM().to(DEVICE)

if torch.cuda.device_count() > 1:
    print(f"\nUsing {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

# ✅ class weights to handle val imbalance
n_neg = (train_df["label"] == 0).sum()
n_pos = (train_df["label"] == 1).sum()
pos_weight = torch.tensor([n_neg / n_pos]).to(DEVICE)
print(f"\nPos weight: {pos_weight.item():.4f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)  # ✅
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# ============================================================
# 🚀 TRAINING LOOP
# ============================================================
best_val_acc = 0

print("\n🚀 Training Started...\n")

for epoch in range(EPOCHS):

    # -------- TRAIN --------
    model.train()
    train_loss = 0

    for frames, labels in train_loader:
        frames = frames.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(frames)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # -------- VALIDATION --------
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for frames, labels in val_loader:
            frames = frames.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(frames)
            preds  = (torch.sigmoid(logits) > 0.5).float()

            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = 100 * correct / total
    f1      = f1_score(all_labels, all_preds)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  💾 Saved best model (acc: {val_acc:.2f}%)")

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.2f}% | F1: {f1:.4f}")

print("\n✅ TRAINING COMPLETE")
print(f"Best Val Acc: {best_val_acc:.2f}%")

Device: cuda
GPUs available: 2

Full dataset distribution:
label
0    3027
1    2958
Name: count, dtype: int64

Train videos: 60 | Val videos: 15
Overlap (should be 0): 0
✅ No leakage confirmed

=== Train label distribution ===
label
1    2502
0    2387
Name: count, dtype: int64

=== Val label distribution ===
label
0    640
1    456
Name: count, dtype: int64

Using 2 GPUs

Pos weight: 0.9540

🚀 Training Started...

  💾 Saved best model (acc: 61.31%)
Epoch 1/20 | Train Loss: 83.6055 | Val Acc: 61.31% | F1: 0.1554
Epoch 2/20 | Train Loss: 28.9570 | Val Acc: 58.39% | F1: 0.0500
Epoch 3/20 | Train Loss: 14.5757 | Val Acc: 59.58% | F1: 0.2187
Epoch 4/20 | Train Loss: 11.4127 | Val Acc: 55.75% | F1: 0.0619
Epoch 5/20 | Train Loss: 9.9545 | Val Acc: 59.22% | F1: 0.0387
  💾 Saved best model (acc: 62.86%)
Epoch 6/20 | Train Loss: 8.6261 | Val Acc: 62.86% | F1: 0.2218
Epoch 7/20 | Train Loss: 7.1284 | Val Acc: 59.12% | F1: 0.0968
  💾 Saved best model (acc: 63.32%)
Epoch 8/20 | Train Loss: 3.687

In [9]:
print("=== Violence CSV columns ===")
print(df_v.columns.tolist())
print(df_v[["video_id", "label", "source"]].head(5))

print("\n=== NonViolence CSV columns ===")
print(df_nv.columns.tolist())
print(df_nv[["video_id", "label", "source"]].head(5))

print("\n=== Unique video_ids (violence) ===")
print(df_v["video_id"].unique())

print("\n=== Unique video_ids (nonviolence) ===")
print(df_nv["video_id"].unique())

print("\n=== Train label distribution ===")
print(train_df["label"].value_counts())

print("\n=== Val label distribution ===")
print(val_df["label"].value_counts())

=== Violence CSV columns ===
['window_id', 'video_id', 'label', 'start_frame', 'end_frame', 'frame_paths', 'source']
   video_id  label    source
0  v_video1      0  violence
1  v_video1      0  violence
2  v_video1      0  violence
3  v_video1      0  violence
4  v_video1      0  violence

=== NonViolence CSV columns ===
['window_id', 'video_id', 'label', 'start_frame', 'end_frame', 'frame_paths', 'source']
    video_id  label       source
0  nv_video1      0  nonviolence
1  nv_video1      0  nonviolence
2  nv_video1      0  nonviolence
3  nv_video1      0  nonviolence
4  nv_video1      0  nonviolence

=== Unique video_ids (violence) ===
['v_video1' 'v_video2' 'v_video3' 'v_video4' 'v_video5' 'v_video6'
 'v_video7' 'v_video8' 'v_video9' 'v_video10' 'v_video11' 'v_video12'
 'v_video13' 'v_video14' 'v_video15' 'v_video16' 'v_video18' 'v_video21'
 'v_video22' 'v_video30' 'v_video31' 'v_video34' 'v_video47']

=== Unique video_ids (nonviolence) ===
['nv_video1' 'nv_video2' 'nv_video3' 'nv_

In [10]:
import os
print(os.listdir("/kaggle/working"))

['best_model.pth', '__notebook__.ipynb']


In [13]:
# ============================================================
# 🚀 EARLY VIOLENCE PREDICTION (CNN + LSTM)
# ============================================================

import os
import ast
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score

# ------------------ CONFIG ------------------
ROOT = "/kaggle/input/datasets/idkabtit/violencepredictiondataset/kaggleDataset"

VIOLENCE_DIR    = os.path.join(ROOT, "violenceFrame")
NONVIOLENCE_DIR = os.path.join(ROOT, "nonViolenceFrame")

CSV_V  = os.path.join(VIOLENCE_DIR, "dataset_kaggle.csv")
CSV_NV = os.path.join(NONVIOLENCE_DIR, "dataset_kaggle.csv")

SAVE_PATH = "/kaggle/working/best_model_arnav_new.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WINDOW_SIZE = 16
BATCH_SIZE  = 16
EPOCHS      = 25
LR          = 1e-4

print("Device:", DEVICE)
print("GPUs available:", torch.cuda.device_count())

# ------------------ TRANSFORMS ------------------
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ============================================================
# 📦 DATASET
# ============================================================
class ViolenceDataset(Dataset):
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        source   = row["source"]
        label    = float(row["label"])
        base_dir = VIOLENCE_DIR if source == "violence" else NONVIOLENCE_DIR

        raw = str(row["frame_paths"])
        if "|" in raw:
            paths = raw.split("|")
        else:
            try:
                paths = ast.literal_eval(raw)
            except:
                paths = raw.split(",")

        frames = []
        for p in paths:
            p         = p.strip()
            full_path = os.path.join(base_dir, p)
            img       = Image.open(full_path).convert("RGB")
            img       = self.transform(img)
            frames.append(img)

        if len(frames) != WINDOW_SIZE:
            raise Exception(f"Window size mismatch at index {idx}")

        return torch.stack(frames), torch.tensor(label, dtype=torch.float32)

# ============================================================
# 📊 LOAD DATA
# ============================================================
df_v  = pd.read_csv(CSV_V);  df_v["source"]  = "violence"
df_nv = pd.read_csv(CSV_NV); df_nv["source"] = "nonviolence"

df_v["video_id"]  = "v_"  + df_v["video_id"].astype(str)
df_nv["video_id"] = "nv_" + df_nv["video_id"].astype(str)

df = pd.concat([df_v, df_nv], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nFull dataset distribution:")
print(df["label"].value_counts())

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df["video_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

assert len(set(train_df["video_id"]) & set(val_df["video_id"])) == 0
print("✅ No leakage confirmed")

print("\n=== Train label distribution ===")
print(train_df["label"].value_counts())
print("\n=== Val label distribution ===")
print(val_df["label"].value_counts())

train_dataset = ViolenceDataset(train_df, transform=transform_train)
val_dataset   = ViolenceDataset(val_df,   transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ============================================================
# 🧠 MODEL
# ============================================================
class CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()

        resnet   = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])

        self.lstm    = nn.LSTM(512, 128, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc      = nn.Linear(128, 1)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x        = x.view(B * T, C, H, W)
        features = self.cnn(x).view(B, T, 512)
        out, _   = self.lstm(features)
        out      = self.dropout(out[:, -1, :])
        return self.fc(out).squeeze(1)

# ============================================================
# ⚙️ SETUP
# ============================================================
model = CNN_LSTM().to(DEVICE)

# freeze CNN
for param in model.cnn.parameters():
    param.requires_grad = False

print("\nCNN frozen ✅")

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

# class weights
n_neg      = (train_df["label"] == 0).sum()
n_pos      = (train_df["label"] == 1).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
print(f"Pos weight: {pos_weight.item():.4f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# only train LSTM + FC
if isinstance(model, nn.DataParallel):
    trainable = list(model.module.lstm.parameters()) + \
                list(model.module.fc.parameters())
else:
    trainable = list(model.lstm.parameters()) + \
                list(model.fc.parameters())

optimizer = torch.optim.Adam(trainable, lr=LR)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

# ============================================================
# 🚀 TRAINING LOOP
# ============================================================
best_val_acc = 0

print("\n🚀 Training Started...\n")

for epoch in range(EPOCHS):

    # -------- TRAIN --------
    model.train()
    train_loss = 0

    for frames, labels in train_loader:
        frames = frames.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(frames)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # -------- VALIDATION --------
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for frames, labels in val_loader:
            frames = frames.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(frames)
            preds  = (torch.sigmoid(logits) > 0.5).float()

            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            all_preds.extend(preds.cpu().numpy().astype(int))
            all_labels.extend(labels.cpu().numpy().astype(int))

    val_acc = 100 * correct / total
    f1      = f1_score(all_labels, all_preds)

    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), SAVE_PATH)
        else:
            torch.save(model.state_dict(), SAVE_PATH)
        print(f"  💾 Saved best model (acc: {val_acc:.2f}%)")

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.2f}% | F1: {f1:.4f}")

print("\n✅ TRAINING COMPLETE")
print(f"Best Val Acc: {best_val_acc:.2f}%")
print(f"Model saved at: {SAVE_PATH}")

Device: cuda
GPUs available: 2

Full dataset distribution:
label
0    3027
1    2958
Name: count, dtype: int64
✅ No leakage confirmed

=== Train label distribution ===
label
1    2502
0    2387
Name: count, dtype: int64

=== Val label distribution ===
label
0    640
1    456
Name: count, dtype: int64

CNN frozen ✅
Using 2 GPUs
Pos weight: 0.9540

🚀 Training Started...

  💾 Saved best model (acc: 59.49%)
Epoch 1/25 | Train Loss: 142.8564 | Val Acc: 59.49% | F1: 0.0939
  💾 Saved best model (acc: 60.86%)
Epoch 2/25 | Train Loss: 86.0035 | Val Acc: 60.86% | F1: 0.1605
Epoch 3/25 | Train Loss: 66.2983 | Val Acc: 59.03% | F1: 0.1315
Epoch 4/25 | Train Loss: 53.9172 | Val Acc: 55.11% | F1: 0.2454
Epoch 5/25 | Train Loss: 45.8093 | Val Acc: 59.03% | F1: 0.1074
Epoch 6/25 | Train Loss: 39.4785 | Val Acc: 47.72% | F1: 0.2900
Epoch 7/25 | Train Loss: 35.7876 | Val Acc: 58.30% | F1: 0.1940
Epoch 8/25 | Train Loss: 29.5223 | Val Acc: 58.76% | F1: 0.1274
Epoch 9/25 | Train Loss: 26.9393 | Val Acc: 5